In [ ]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq xcube  # upgrade xcube on colab

In [ ]:
#| default_exp l2r.models.core

In [ ]:
#| export
import logging
import os
from typing import Optional, Dict
from fastai.torch_imports import *
from fastai.layers import *
# from xcube.layers import *
from transformers import AutoTokenizer, PretrainedConfig, PreTrainedModel, AutoConfig, AutoModel, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from safetensors.torch import load_file as load_safetensors
from peft import LoraConfig, get_peft_model
import sys
from io import StringIO
import io
import logging
from contextlib import redirect_stdout
from huggingface_hub import hf_hub_download
from xcube.utils import compute_trainable_params 

In [8]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

# L2R Models

> Contains the models for learning to rank

## Linear

In [9]:
#| export
class L2R_DotProductBias(nn.Module):
    def __init__(self, num_lbs, num_toks, num_factors, y_range=None):
        super().__init__()
        self.num_toks, self.num_lbs = num_toks+1, num_lbs+1 # +1 for the `padding_idx` 
        self.num_factors = num_factors
        self.token_factors = nn.Embedding(self.num_toks, num_factors, padding_idx=-1)
        self.token_bias = nn.Embedding(self.num_toks, 1, padding_idx=-1)
        self.label_factors = nn.Embedding(self.num_lbs, num_factors, padding_idx=-1)
        self.label_bias = nn.Embedding(self.num_lbs, 1, padding_idx=-1)
        self.y_range = y_range
        
    def forward(self, xb):
        # import pdb; pdb.set_trace()
        xb_toks = xb[:, :, :, 0].long() # xb[...,0] # shape (64, 2233, 64)
        xb_lbs = torch.unique(xb[:, :, :, 1], dim=-1).flatten(start_dim=1).long() # shape (64, 2233, )
        # To convert -1 which is the padding index to the last index:
        xb_toks, xb_lbs= xb_toks%(self.num_toks), xb_lbs%(self.num_lbs)
        
        toks_embs = self.token_factors(xb_toks) # shape (64, 2233, 64, 400)
        toks_shape = toks_embs.shape
        toks_embs = toks_embs.view(-1, *toks_shape[2:]) # shape (64*2233, 64, 400)

        lbs_embs = self.label_factors(xb_lbs) # shape (64, 2233, 400)
        lbs_shape = lbs_embs.shape
        lbs_embs = lbs_embs.view(-1, *lbs_shape[2:]).unsqueeze(dim=-1) # shape (64*2233, 400, 1)
        
        # res = torch.bmm(toks_embs, lbs_embs) # shape (64*2233, 64, 1)
        # res = res.view(toks_shape[0], toks_shape[1], *res.shape[1:]) + self.token_bias(xb_toks) + self.label_bias(xb_lbs).unsqueeze(2) # shape (65, 2233, 64, 1)

        bias = self.token_bias(xb_toks) + self.label_bias(xb_lbs).unsqueeze(2) # (bs, lbs_in_one_chunk, sl, 1)
        bias = bias.view(-1, *bias.shape[2:]) # (bs*lbs_in_one_chunk, sl, 1)
        res = torch.baddbmm(bias, toks_embs, lbs_embs) # toks_embs: (bs*lbs_in_one_chunk, sl, emb_sz), lbs_embs: (bs*lbs_in_one_chunk, emb_sz, 1), res : (bs*lbs_in_one_chunk, sl, 1)
        res = res.view(toks_shape[0], toks_shape[1], *res.shape[1:]) # res (bs, lbs_in_one_chunk, sl, 1)
        
        return sigmoid_range(res, *self.y_range) if self.y_range is not None else res
        # return res

## Neural Network

In [10]:
#| export
class L2R_NN(nn.Module):
    def __init__(self, num_lbs, 
                 num_toks, 
                 num_factors, 
                 layers,
                 ps=None,
                 use_bn=True,
                 bn_final=False,
                 lin_first=True,
                 embed_p=0.0,
                 y_range=None):
        super().__init__()
        self.num_toks, self.num_lbs = num_toks+1, num_lbs+1 # +1 for the `padding_idx` 
        self.num_factors, self.embed_p, self.ps = num_factors, embed_p, ps
        self.token_factors = nn.Embedding(self.num_toks, num_factors, padding_idx=-1)
        self.label_factors = nn.Embedding(self.num_lbs, num_factors, padding_idx=-1)
        self.emb_drop = nn.Dropout(embed_p)
        self.y_range = y_range
        if ps is None: self.ps = [0. for _ in range(len(layers))]
        sizes = [self.token_factors.embedding_dim] + layers + [1]
        actns = [nn.ReLU(inplace=True) for _ in range(len(sizes)-2)] + [None]
        _layers = [LinBnFlatDrop(sizes[i], sizes[i+1], bn=use_bn and (i!=len(actns)-1 or bn_final), p=p, act=a, lin_first=lin_first)
                   for i, (p,a) in enumerate(zip(self.ps+[0.],actns))]
        self.layers= nn.Sequential(*_layers)
        # self.layers = nn.Sequential(
        #     nn.Linear(num_factors, n_act),
        #     nn.ReLU(),
        #     nn.Linear(n_act, 1),
        #     nn.Dropout(self.ps),
        # )
        
    # def __str__(self): return super().__repr__() + f"\n {self.n_act = }, {self.embed_p = }"
    # __repr__ = __str__

    
    def forward(self, xb):
        # import pdb; pdb.set_trace()
        xb_toks = xb[:, :, :, 0].long() # xb[...,0] # shape (64, 2233, 64)
        xb_lbs = torch.unique(xb[:, :, :, 1], dim=-1).flatten(start_dim=1).long() # shape (64, 2233, )
        # To convert -1 which is the padding index to the last index:
        xb_toks, xb_lbs= xb_toks%(self.num_toks), xb_lbs%(self.num_lbs)
        
        toks_embs = self.token_factors(xb_toks) # shape (64, 2233, 64, 200)

        lbs_embs = self.label_factors(xb_lbs) # shape (64, 2233, 200)
        lbs_embs = lbs_embs.unsqueeze(2) # shape (64, 2233, 1, 200)
        lbs_embs = lbs_embs.expand(-1, -1, xb.shape[2], -1)
        
        # embs = torch.cat((toks_embs, lbs_embs), dim=-1) # shape (64, 2233, 64, 400)
        embs = toks_embs + lbs_embs
        embs = self.emb_drop(embs)
        res = self.layers(embs)
        
        return sigmoid_range(res, *self.y_range) if self.y_range is not None else res
        # return res

## Using Hugging Face Base Model

In [ ]:
# | export
def reshape_relevance_scores(predicted_relevance_scores, true_relevance_scores):
    """
    Reshapes predicted and true relevance score tensors for relevance ranking tasks.

    Args:
        predicted_relevance_scores (torch.Tensor): Predicted relevance scores, shape (batch_size, max_len, num_labels)
        true_relevance_scores (torch.Tensor): Ground-truth relevance scores, shape (batch_size, max_len, num_labels)

    Returns:
        tuple: (reshaped_pred_scores, reshaped_true_scores) with shape (batch_size * num_labels, max_len) for each,
               where batch and label dimensions are combined for per-label sequence processing
    """
    batch_size, sequence_length, num_labels = predicted_relevance_scores.shape

    # Reshape to combine batch and label dimensions for per-label sequence processing
    reshaped_pred_scores = predicted_relevance_scores.permute(0, 2, 1).reshape(
        batch_size * num_labels, sequence_length
    )
    reshaped_true_scores = true_relevance_scores.permute(0, 2, 1).reshape(
        batch_size * num_labels, sequence_length
    )

    return reshaped_pred_scores, reshaped_true_scores

def log_print_output(func, *args, prefix="📊", **kwargs):
    """
    Capture print output from any function and log it

    Args:
        func: The function to call
        *args: Positional arguments to pass to the function
        prefix: Emoji/prefix for the log message
        **kwargs: Keyword arguments to pass to the function

    # Usage examples:
    log_print_output(model.print_trainable_parameters)
    log_print_output(print, "Hello World", prefix="🔍")
    log_print_output(some_function, arg1, arg2, keyword_arg=value, prefix="⚡")
    """
    with io.StringIO() as buf, redirect_stdout(buf):
        func(*args, **kwargs)
        output = buf.getvalue().strip()
        if output:  # Only log if there's actual output
            logging.info(f"{prefix} {output}")

In [ ]:
# | export
from ptranking.ltr_adhoc.listwise.approxNDCG import approxNDCG_loss
from ptranking.ltr_adhoc.eval.eval_utils import LABEL_TYPE
from torch.nn.init import trunc_normal_

In [ ]:
# | export
class LTRConfig(PretrainedConfig):
    model_type = "ltr_model"

    def __init__(self, num_labels=7942, ground_model_name_or_path=None, **kwargs):
        super().__init__(**kwargs)
        # Register additional attributes
        self.num_labels = num_labels
        self.ground_model_name_or_path = (
            ground_model_name_or_path or "mistralai/Mistral-7B-Instruct-v0.3"
        )


class LTRModel(PreTrainedModel):
    """
    Learning-to-Rank model based on a pretrained transformer, outputting per-token relevance scores per label
    using label embeddings and matrix multiplication.

    Args:
        base_model (AutoModelForCausalLM): Pretrained transformer model.
        num_labels (int): Number of labels for relevance scoring.
    """

    config_class = LTRConfig

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.hidden_size = config.hidden_size
        # Label embeddings
        self.label_embeddings = nn.Embedding(self.num_labels, self.hidden_size)
        # Initialize its weights using truncated normal initialization
        trunc_normal_(self.label_embeddings.weight, std=0.01)

        # Initialize ground model as None (will be set in from_pretrained)
        self.ground_model = None
        # Initialize lookup table
        self.lookup_table = None

    def _init_ground_model(self, config, quantization_config=None):
        """Initialize the ground model with pretrained weights and apply LoRA."""
        try:
            model_name = config.ground_model_name_or_path

            # Determine model type and load appropriately
            logging.info(f"🧠 Loading ground model {model_name} with quantization...")
            if "bert" in model_name.lower():
                # BERT-based models use AutoModel
                self.ground_model = AutoModel.from_pretrained(
                    model_name,
                    quantization_config=quantization_config,
                    device_map="auto",
                    trust_remote_code=True,
                    torch_dtype=torch.bfloat16,
                )
            else:
                # Default to causal LM for models like Mistral
                self.ground_model = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config=quantization_config,
                    device_map="auto",
                    trust_remote_code=True,
                    torch_dtype=torch.bfloat16,
                )
            logging.info(f"✅ Ground model loaded from {model_name}")

            # Define default LoRA configuration based on model type
            lora_config = None
            logging.info("🔧 Setting up default LoRA configuration...")
            if "bert" in model_name.lower():
                # LoRA config for BERT-based models
                lora_config = LoraConfig(
                    r=16,
                    lora_alpha=32,
                    target_modules=["query", "key", "value", "output.dense"],
                    lora_dropout=0.05,
                    bias="none",
                    task_type="MASKED_LM",
                )
            else:
                # LoRA config for causal LM models
                lora_config = LoraConfig(
                    r=16,
                    lora_alpha=32,
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                    lora_dropout=0.05,
                    bias="none",
                    task_type="CAUSAL_LM",
                )

            # Log LoRA configuration
            logging.info(
                f"🔍 LoRA config: r={lora_config.r}, alpha={lora_config.lora_alpha}, "
                f"targets={lora_config.target_modules}, dropout={lora_config.lora_dropout}"
            )

            # Apply LoRA
            logging.info("🧩 Applying LoRA adapters to ground model...")
            self.ground_model = get_peft_model(self.ground_model, lora_config)
            log_print_output(func=self.ground_model.print_trainable_parameters)
            log_print_output(compute_trainable_params, self.ground_model)

        except Exception as e:
            logging.error(f"❌ Error initializing ground model: {e}")
            raise RuntimeError(f"Ground model initialization failed: {e}")

    def forward_old(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """
        Forward pass to compute per-token relevance scores per label.

        Args:
            input_ids (torch.Tensor): Shape (batch_size, max_len).
            attention_mask (torch.Tensor): Shape (batch_size, max_len).

        Returns:
            torch.Tensor: Relevance scores of shape (batch_size, max_len, num_labels).
        """
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # hidden_states = outputs.hidden_states[-1]  # Shape: (batch_size, max_len, hidden_size)
        hidden_states = (
            outputs.last_hidden_state
        )  # Shape: (batch_size, max_len, hidden_size)
        relevance_scores = self.relevance_head(
            hidden_states
        )  # Shape: (batch_size, max_len, num_labels)
        return relevance_scores

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass to compute per-token relevance scores per label using matrix multiplication
        between token hidden states and label embeddings.

        Args:
            input_ids (torch.Tensor): Shape (batch_size, max_len).
            attention_mask (torch.Tensor): Shape (batch_size, max_len).
            labels (torch.Tensor, optional): Ground-truth relevance scores, shape (batch_size, max_len, num_labels).

        Returns:
            torch.Tensor: Relevance scores of shape (batch_size, max_len, num_labels).
        """
        if self.ground_model is None:
            raise ValueError(
                "ground_model is not initialized. Ensure it is set before calling forward."
            )

        outputs = self.ground_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        hidden_states = outputs.hidden_states[
            -1
        ]  # Shape: (batch_size, max_len, hidden_size) # mistrals
        # hidden_states = outputs.last_hidden_state  # Shape: (batch_size, max_len, hidden_size) # bert-based

        # Create label indices for all labels
        batch_size = hidden_states.size(0)
        label_indices = (
            torch.arange(self.num_labels, device=hidden_states.device)
            .unsqueeze(0)
            .expand(batch_size, -1)
        )  # (batch_size, num_labels)
        label_queries = self.label_embeddings(
            label_indices
        )  # (batch_size, num_labels, hidden_size)

        # Compute relevance scores via matrix multiplication
        # hidden_states: (batch_size, max_len, hidden_size)
        # label_queries.transpose: (batch_size, hidden_size, num_labels)
        # Result: (batch_size, max_len, num_labels)
        predicted_relevance_scores = torch.bmm(
            hidden_states, label_queries.transpose(1, 2)
        )  # Shape: (batch_size, max_len, num_labels)

        output_dict = {"logits": predicted_relevance_scores}

        # Add hidden_states
        output_dict["hidden_states"] = hidden_states

        if self.lookup_table is not None:
            # Ground-truth relevance scores, shape (batch_size, max_len, num_labels)
            true_relevance_scores = self.lookup_table[input_ids, :].to(hidden_states.device)
            output_dict["labels"] = true_relevance_scores

            # Compute loss
            # loss1 = lambdarank_loss(predicted_relevance_scores, true_relevance_scores)
            predicted_relevance_scores, true_relevance_scores = reshape_relevance_scores(
                predicted_relevance_scores, true_relevance_scores
            )
            # true_relevance_scores are fine-grained quantile-bin grades (0-101,
            # see quantize_l2r_data), not the handful of LETOR-style relevance
            # levels LABEL_TYPE.MultiLabel's exponential gain (2^rel - 1) assumes.
            # LABEL_TYPE.Permutation uses linear gains (batch_stds as-is), which
            # stays numerically stable across the full grade range.
            loss = approxNDCG_loss(
                batch_preds=predicted_relevance_scores,
                batch_stds=true_relevance_scores,
                label_type=LABEL_TYPE.Permutation,
                gpu=torch.cuda.is_available(),
            )
            # import ipdb; ipdb.set_trace()
            output_dict["loss"] = loss

        return output_dict

    def compute_trainable_params(self):
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        trainable_percentage = 100 * trainable_params / total_params

        logging.info(
            f"📊 trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {trainable_percentage:.4f}"
        )
        return trainable_params, total_params

    def save_pretrained(self, save_directory, **kwargs):
        """Override save_pretrained to include ground_model and lookup_table."""
        os.makedirs(save_directory, exist_ok=True)
        super().save_pretrained(save_directory, **kwargs)
        if self.ground_model is not None:
            self.ground_model.save_pretrained(
                os.path.join(save_directory, "ground_model")
            )
        if self.lookup_table is not None:
            torch.save(
                self.lookup_table, os.path.join(save_directory, "lookup_table.pt")
            )
        logging.info(
            f"✅ Model, ground_model, and lookup_table saved to {save_directory}"
        )

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, *model_args, device_map="cpu", **kwargs):
        """Override from_pretrained to load weights from model.safetensors or pytorch_model.bin."""

        device = "cuda" if torch.cuda.is_available() else "cpu"

        config = LTRConfig.from_pretrained(
            pretrained_model_name_or_path, *model_args, **kwargs
        )
        model = cls(config)

        # Initialize ground model with LoRA
        # quantization_config = kwargs.get("quantization_config", None)
        model._init_ground_model(config, quantization_config=config.quantization_config)

        # Move model to specified device
        model = model.to(device)
        logging.info(f"🖥️ Model moved to device: {device}")

        # Load from a local directory if given one; otherwise download from the Hub.
        # hf_hub_download requires a namespace/repo_name id and raises HFValidationError
        # on a local path, so that branch must never be taken for on-disk checkpoints
        # (e.g. Stage 4's local L2R output loaded as Stage 5's --hub_model_id).
        custom_state_dict = None
        hf_token = kwargs.get("token", None)
        is_local = os.path.isdir(pretrained_model_name_or_path)

        safetensors_path = os.path.join(pretrained_model_name_or_path, "model.safetensors")
        try:
            if is_local:
                if not os.path.exists(safetensors_path):
                    raise FileNotFoundError(safetensors_path)
            else:
                safetensors_path = hf_hub_download(
                    repo_id=pretrained_model_name_or_path,
                    filename="model.safetensors",
                    token=hf_token,
                )
            custom_state_dict = load_safetensors(safetensors_path)
            logging.info(f"✅ Loaded model weights from {safetensors_path}")
        except Exception as e:
            logging.warning(
                f"❌ Failed to load model.safetensors: {e}, trying pytorch_model.bin"
            )

        if custom_state_dict is None:
            pytorch_bin_path = os.path.join(pretrained_model_name_or_path, "pytorch_model.bin")
            try:
                if is_local:
                    if not os.path.exists(pytorch_bin_path):
                        raise FileNotFoundError(pytorch_bin_path)
                else:
                    pytorch_bin_path = hf_hub_download(
                        repo_id=pretrained_model_name_or_path,
                        filename="pytorch_model.bin",
                        token=hf_token,
                    )
                custom_state_dict = torch.load(pytorch_bin_path, map_location="cpu")
                logging.info(f"✅ Loaded model weights from {pytorch_bin_path}")
            except Exception as e:
                logging.error(f"❌ Failed to load pytorch_model.bin: {e}")
                raise

        if custom_state_dict is None:
            logging.error(
                f"❌ No valid model weights found in {pretrained_model_name_or_path}"
            )
            raise FileNotFoundError(
                f"No model.safetensors or pytorch_model.bin found in {pretrained_model_name_or_path}"
            )

        # Move tensors to the specified device
        custom_state_dict = {k: v.to(device) for k, v in custom_state_dict.items()}

        # Align state_dict keys with model's expected order
        model_state_dict = model.state_dict()
        if set(custom_state_dict.keys()) != set(model_state_dict.keys()):
            logging.warning(
                f"⚠️ Key mismatch: loaded keys {set(custom_state_dict.keys())} != model keys {set(model_state_dict.keys())}"
            )
        ordered_state_dict = {
            k: custom_state_dict[k]
            for k in model_state_dict.keys()
            if k in custom_state_dict
        }

        # Load weights
        model.load_state_dict(ordered_state_dict, strict=False)
        logging.info(f"✅ Custom weights loaded into model")

        # Verify weights match
        weights_match = True
        for key in list(model_state_dict.keys())[:2]:  # Check first two parameters
            if key in custom_state_dict and not torch.equal(
                custom_state_dict[key], model_state_dict[key]
            ):
                logging.warning(f"⚠️ Weight mismatch for {key}")
                weights_match = False
        if weights_match:
            logging.info(f"✔️ Loaded weights match model's state_dict")
        else:
            logging.warning(f"⚠️ Some loaded weights do not match model's state_dict")

        # Load lookup table
        try:
            lookup_table_path = os.path.join(pretrained_model_name_or_path, "lookup_table.pt")
            if is_local:
                if not os.path.exists(lookup_table_path):
                    raise FileNotFoundError(lookup_table_path)
            else:
                lookup_table_path = hf_hub_download(
                    repo_id=pretrained_model_name_or_path,
                    filename="lookup_table.pt",
                    token=hf_token,
                )
            model.lookup_table = torch.load(lookup_table_path, map_location=device)
            logging.info(f"✅ Lookup table loaded from {lookup_table_path}")
        except Exception as e:
            logging.error(
                f"❌ Error loading lookup table from {pretrained_model_name_or_path}: {e}"
            )
            raise

        logging.info("🧩 After loading the pretrained weights...")
        log_print_output(func=model.compute_trainable_params)

        return model

## Export -

In [11]:
#| hide
import nbdev; nbdev.nbdev_export()